<a href="https://colab.research.google.com/github/APES-M-JeissonS-JuanB/apes-pobreza-multidimensional-ecv2023/blob/main/notebooks/APES_Entrega_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pobreza multidimensional en Colombia: factores asociados según la ECV 2023
**APES – Entrega final** · Jeisson Sánchez y Juan Bogotá

Fuente: Encuesta Nacional de Calidad de Vida (ECV) 2023, DANE. Unidad de análisis: hogar y persona (llave `DIRECTORIO` + `SECUENCIA_P` [+ `ORDEN`]).

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APES-M-JeissonS-JuanB/apes-pobreza-multidimensional-ecv2023/blob/main/notebooks/APES_Entrega_Final.ipynb)



## 0. SETUP

In [1]:
# SETUP
# Qué hace: instala dependencias, importa las librerías, fija opciones de visualización y define las rutas del proyecto.
# Por qué: todas las rutas son relativas a la raíz del repo, así el notebook corre igual en el computador de cualquiera de los dos.
%pip install -q openpyxl pyarrow

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Opciones de presentación: sin advertencias, hasta 100 columnas y 2 decimales con separador de miles.
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
# Semilla fija por reproducibilidad.
np.random.seed(42)

# Ruta base: raíz del repo (el notebook vive en notebooks/)
# El notebook vive en notebooks/, así que la raíz del repo es la carpeta superior (..).
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Carpetas del proyecto: CSV originales (data/raw/csv), bases limpias (data/processed) y gráficos (figures).
RAW_CSV   = os.path.join(BASE_PATH, "data", "raw", "csv")
PROCESSED = os.path.join(BASE_PATH, "data", "processed")
FIGURES   = os.path.join(BASE_PATH, "figures")
# Crea las carpetas de salida si no existen.
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

# Verificación: debe mostrar la raíz del repo y los 11 CSV de la ECV.
print("BASE_PATH:", BASE_PATH)
print("CSV encontrados:", sorted(os.listdir(RAW_CSV)))

BASE_PATH: /


FileNotFoundError: [Errno 2] No such file or directory: '/data/raw/csv'

## 1. CARGA DE DATOS

In [ ]:
# CARGA DE LOS 11 MÓDULOS DE LA ECV
# Qué hace: lee cada CSV del DANE en un DataFrame y los guarda en el diccionario dfs.
# Por qué: cada tema (personas, vivienda, salud...) viene en un archivo distinto y después se unen por llave.
# Nombre corto de cada módulo -> archivo CSV en data/raw/csv.
ARCHIVOS = {
    "personas":        "personas.csv",
    "vivienda":        "vivienda.csv",
    "servicios":       "servicios.csv",
    "condiciones":     "condiciones.csv",
    "educacion":       "educacion.csv",
    "fuerza_trabajo":  "fuerza_trabajo.csv",
    "salud":           "salud.csv",
    "diseno_muestra":  "diseno_muestra.csv",
    "trabajo_infantil":"trabajo_infantil.csv",
    "primera_infancia":"primera_infancia.csv",
    "tic":             "tic.csv",
}

# Función auxiliar: detecta el separador (coma o punto y coma) y la codificación (utf-8 o latin-1) mirando la primera línea.
def leer_csv(ruta):
    """Lee un CSV del DANE probando separador y codificación."""
    with open(ruta, "rb") as f:
        primera = f.readline()
    for enc in ("utf-8", "latin-1"):
        try:
            linea = primera.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    sep = ";" if linea.count(";") > linea.count(",") else ","
    return pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False)

# Carga los 11 archivos y estandariza los nombres de columna (sin espacios y en mayúsculas).
dfs = {}
for nombre, archivo in ARCHIVOS.items():
    dfs[nombre] = leer_csv(os.path.join(RAW_CSV, archivo))
    dfs[nombre].columns = dfs[nombre].columns.str.strip().str.upper()

# Resumen de dimensiones
# Tabla de control: filas y columnas de cada módulo.
resumen = pd.DataFrame({
    "filas":    {k: v.shape[0] for k, v in dfs.items()},
    "columnas": {k: v.shape[1] for k, v in dfs.items()},
})
resumen

In [ ]:
# VERIFICACIÓN DE LLAVES Y TAMAÑOS
# Qué hace: comprueba que todos los módulos tengan las columnas llave y que el número de hogares y personas coincida con la ficha técnica.
# Llaves: hogar = DIRECTORIO + SECUENCIA_P; persona = hogar + ORDEN.
# Definición de las llaves de unión (se usan en todo el notebook).
LLAVE_HOGAR = ["DIRECTORIO", "SECUENCIA_P"]
LLAVE_PERSONA = LLAVE_HOGAR + ["ORDEN"]

# ¿Cada módulo trae las columnas de la llave de hogar y de la llave de persona?
for nombre, df in dfs.items():
    tiene_hogar   = all(c in df.columns for c in LLAVE_HOGAR)
    tiene_persona = all(c in df.columns for c in LLAVE_PERSONA)
    print(f"{nombre:18s} llave hogar: {tiene_hogar!s:5} | llave persona: {tiene_persona!s:5}")

# Conteos esperados según la ficha técnica: 86.405 hogares y 240.212 personas.
n_hogares  = dfs["personas"][LLAVE_HOGAR].drop_duplicates().shape[0]
n_personas = dfs["personas"].shape[0]
print(f"\nHogares en 'personas':  {n_hogares:,}  (esperado 86.405)")
print(f"Personas en 'personas': {n_personas:,}  (esperado 240.212)")

In [ ]:
# VISTA RÁPIDA
# Qué hace: muestra las primeras 3 filas (10 primeras columnas) de cada módulo para reconocer su estructura.
for nombre, df in dfs.items():
    print(f"\n=== {nombre} {df.shape} ===")
    display(df.head(3).iloc[:, :10])

**Conclusión.** Los 11 módulos cargan y las llaves están en todos. `personas` tiene 86.405 hogares y 240.212 personas, igual que la ficha técnica.

## 2. INTEGRACIÓN
### 2.1 Duplicados por llave en cada módulo

In [ ]:
# 2.1 DUPLICADOS POR LLAVE
# Qué hace: para cada módulo cuenta las filas repetidas según la llave de hogar y la de persona.
# Cómo leerlo: dup_llave_persona = 0 significa que la llave de persona identifica una fila única.
# dup_llave_hogar > 0 es normal en módulos de personas (varias personas por hogar) pero no en módulos de hogar.
filas = []
for nombre, df in dfs.items():
    filas.append({
        "modulo": nombre,
        "filas": len(df),
        "dup_llave_hogar": df.duplicated(LLAVE_HOGAR).sum(),
        "dup_llave_persona": df.duplicated(LLAVE_PERSONA).sum(),
        "ORDEN_unicos": df["ORDEN"].nunique(),
    })
pd.DataFrame(filas).set_index("modulo")

**Conclusión.** En ningún módulo se repite la llave de persona (`DIRECTORIO` + `SECUENCIA_P` + `ORDEN`). `servicios` y `condiciones` tienen 342 llaves de hogar repetidas y `vivienda` tiene una fila por vivienda (86.063), así que en esos módulos la llave de hogar no es la misma que en `personas`.

### 2.2 ¿Cuántos hogares de 'personas' no aparecen en cada módulo de hogar?

In [ ]:
# 2.2 HOGARES SIN REGISTRO EN LOS MÓDULOS DE HOGAR
# Qué hace: cruza la lista de hogares (tomada de 'personas') con vivienda, servicios y condiciones usando DIRECTORIO + SECUENCIA_P.
# Resultado: 342 hogares no aparecen, es decir, la llave de esos módulos no coincide con la de personas.
# Lista única de hogares (86.405) tomada del módulo de personas.
hogares = dfs["personas"][LLAVE_HOGAR].drop_duplicates()
# merge con indicator=True: '_merge == left_only' marca los hogares que no encuentran pareja en el módulo.
for m in ["vivienda", "servicios", "condiciones"]:
    cruce = hogares.merge(dfs[m][LLAVE_HOGAR].drop_duplicates(), on=LLAVE_HOGAR, how="left", indicator=True)
    print(f"{m:12s} hogares sin registro: {(cruce['_merge'] == 'left_only').sum():,}")

# Alternativa: ¿vivienda se une solo por DIRECTORIO?
sin_dir = hogares[~hogares["DIRECTORIO"].isin(dfs["vivienda"]["DIRECTORIO"])]
print(f"\nHogares cuyo DIRECTORIO no está en vivienda: {len(sin_dir):,}")

**Conclusión.** Con `DIRECTORIO` + `SECUENCIA_P`, 342 hogares quedan sin registro en `vivienda`, `servicios` y `condiciones`. Todos los `DIRECTORIO` sí están en `vivienda`, por lo que `vivienda` se une solo por `DIRECTORIO`.

### 2.3 Buscar variables clave en todos los módulos

In [ ]:
# 2.3 BÚSQUEDA DE VARIABLES CLAVE
# Qué hace: busca por texto en los nombres de columna de todos los módulos (IPM, pobreza, ingreso, factor de expansión, zona, región...).
# Resultado: el IPM NO viene calculado (hay que construirlo); el ingreso está en servicios; FEX_C está en todos los módulos.
palabras = ["IPM", "POBREZ", "INGRES", "I_HOGAR", "PERCAPITA", "FEX", "FACTOR",
            "CLASE", "REGION", "DOMINIO", "DEPARTAMENTO", "P6040", "P6020", "P6050", "P6008"]

for p in palabras:
    hallazgos = {m: [c for c in df.columns if p in c] for m, df in dfs.items()}
    hallazgos = {m: c for m, c in hallazgos.items() if c}
    print(f"\n[{p}]")
    for m, cols in hallazgos.items():
        print(f"   {m}: {cols[:8]}")

**Conclusión.** El microdato no trae el IPM ni una variable de pobreza: hay que **construirlo** (Sección 4). El ingreso (`I_HOGAR`, `PERCAPITA`) está en `servicios`; el factor de expansión `FEX_C` está en todos los módulos; `CLASE`, `REGION` y `P1_DEPARTAMENTO` están en `vivienda`; edad (`P6040`) y sexo (`P6020`) están en `personas`.

### 2.4 Bases integradas (personas y hogares)

In [ ]:
# 2.4 INTEGRACIÓN: BASE DE PERSONAS Y BASE DE HOGARES
# Qué hace: une los módulos en dos bases: df_personas (una fila por persona) y df_hogares (una fila por hogar).
# Por qué: las variables del IPM están repartidas en varios módulos.
# Función de unión: left merge que agrega solo las columnas que la base aún no tiene (evita columnas duplicadas).
def unir(base, df, llave, how="left"):
    """Une df a base, agregando solo columnas que base aún no tiene."""
    nuevas = [c for c in df.columns if c not in base.columns or c in llave]
    return base.merge(df[nuevas].drop_duplicates(llave), on=llave, how=how)



# --- Base de personas: parte de 'personas' y le agrega salud, diseño muestral, educación, trabajo, TIC,
# trabajo infantil y primera infancia, todos con la llave de persona.
df_personas = dfs["personas"].copy()
for m in ["salud", "diseno_muestra", "educacion", "fuerza_trabajo", "tic",
          "trabajo_infantil", "primera_infancia"]:
    df_personas = unir(df_personas, dfs[m], LLAVE_PERSONA)
# Control: un left merge no debe perder ni duplicar personas.
assert len(df_personas) == 240212, "Se perdieron o duplicaron personas"


# --- Base de hogares. En servicios, condiciones y vivienda SECUENCIA_P vale siempre 1 y ORDEN es el número de hogar;
# por eso se renombra ORDEN como SECUENCIA_P para poder unir con la llave de hogar (ver 2.5).
def a_llave_hogar(df):
    return df.drop(columns=["SECUENCIA_P"]).rename(columns={"ORDEN": "SECUENCIA_P"})

# Se parte de los 86.405 hogares y se agregan servicios (incluye el ingreso) y condiciones.
df_hogares = hogares.copy()
df_hogares = unir(df_hogares, a_llave_hogar(dfs["servicios"]), LLAVE_HOGAR)
df_hogares = unir(df_hogares, a_llave_hogar(dfs["condiciones"]), LLAVE_HOGAR)

# vivienda: una fila por vivienda, se une solo por DIRECTORIO
viv = dfs["vivienda"].drop(columns=["SECUENCIA_P", "ORDEN"], errors="ignore")
df_hogares = unir(df_hogares, viv, ["DIRECTORIO"])

# Tamaño del hogar
# Tamaño del hogar = número de personas por hogar, contado desde la base de personas.
tam = df_personas.groupby(LLAVE_HOGAR).size().rename("TAM_HOGAR").reset_index()
df_hogares = df_hogares.merge(tam, on=LLAVE_HOGAR, how="left")
# Control: deben quedar exactamente 86.405 hogares.
assert len(df_hogares) == 86405, "Se perdieron o duplicaron hogares"

# Verificaciones: dimensiones finales, nulos en variables clave y coherencia del tamaño del hogar con CANT_PERSONAS_HOGAR.
print("df_personas:", df_personas.shape)
print("df_hogares: ", df_hogares.shape)
for c in ["I_HOGAR", "PERCAPITA", "CLASE", "TAM_HOGAR"]:
    print(f"{c} nulo:", df_hogares[c].isna().sum())
print("TAM_HOGAR != CANT_PERSONAS_HOGAR:",
      (df_hogares["TAM_HOGAR"] != df_hogares["CANT_PERSONAS_HOGAR"]).sum())

**Conclusión.** `df_personas` queda con 240.212 filas y `df_hogares` con 86.405. `I_HOGAR`, `PERCAPITA`, `CLASE` y `TAM_HOGAR` no tienen nulos, y el tamaño calculado coincide con `CANT_PERSONAS_HOGAR`. Se conservan todas las columnas por ahora; en la Sección 4.2 se guardan solo las que se usan.

### 2.5 Llave de hogar en servicios y condiciones

In [ ]:
# 2.5 CONFIRMAR LA LLAVE DE HOGAR EN SERVICIOS Y CONDICIONES
# Qué hace: prueba la hipótesis de que en esos módulos el hogar se identifica con DIRECTORIO + ORDEN.
# Resultado: con esa llave quedan 0 hogares sin registro, así que la hipótesis se confirma.
cols_id = lambda df: [c for c in df.columns if "SECUENCIA" in c or c in ("DIRECTORIO", "ORDEN")]
print("servicios:", cols_id(dfs["servicios"]))
print("personas: ", cols_id(dfs["personas"]))
print("personas SECUENCIA_P:", sorted(dfs["personas"]["SECUENCIA_P"].unique())[:10])
print("servicios SECUENCIA_P:", sorted(dfs["servicios"]["SECUENCIA_P"].unique())[:10])
print("servicios ORDEN:", sorted(dfs["servicios"]["ORDEN"].unique()))

# Hipótesis: en servicios/condiciones el hogar es DIRECTORIO + ORDEN
for m in ["servicios", "condiciones"]:
    der = dfs[m][["DIRECTORIO", "ORDEN"]].rename(columns={"ORDEN": "SECUENCIA_P"})
    c = hogares.merge(der, on=LLAVE_HOGAR, how="left", indicator=True)
    print(f"{m}: hogares sin registro con DIRECTORIO+ORDEN = {(c['_merge'] == 'left_only').sum():,}")

# Verificación contra la "Estructura de base de datos ECV 2023" (DANE):
# en las tablas de hogar y de personas, ORDEN debe coincidir con SECUENCIA_ENCUESTA.
for m in ["servicios", "condiciones", "personas", "salud", "educacion"]:
    d = dfs[m]
    print(f"{m:12s} ORDEN == SECUENCIA_ENCUESTA en {100 * (d['ORDEN'] == d['SECUENCIA_ENCUESTA']).mean():.2f}% de las filas")

**Conclusión.** Según la *Estructura de base de datos ECV 2023* del DANE, en las tablas de hogar `SECUENCIA_ENCUESTA` identifica el hogar dentro de la vivienda y `SECUENCIA_P` vale siempre 1; en las tablas de personas, `SECUENCIA_P` identifica el hogar. En los datos, `ORDEN` coincide con `SECUENCIA_ENCUESTA` en el 100% de las filas, así que unir por `DIRECTORIO` + `ORDEN` equivale a la llave oficial. Con ella los 342 hogares faltantes aparecen (0 sin registro), y por eso esos módulos se unen con `a_llave_hogar()`. `vivienda` se une solo por `DIRECTORIO`.

### 2.6 Parentesco, tamaño del hogar y estrato

In [ ]:
# 2.6 PARENTESCO, TAMAÑO DEL HOGAR, ESTRATO Y DICCIONARIO
# Qué hace: (1) busca columnas por nombre, (2) carga el diccionario de datos del DANE y define dos funciones de consulta: buscar_dic() y ver().
# Resultado: P6051 es el parentesco (1 = jefe/a); ESTRATO2020 es de diseño muestral, no el estrato socioeconómico.
for pal in ["PARENT", "JEFE", "P6051", "CANT", "PERSONAS", "ESTRATO", "ZONA", "AREA"]:
    print(f"\n[{pal}]")
    for m, df in dfs.items():
        cols = [c for c in df.columns if pal in c]
        if cols:
            print(f"   {m}: {cols[:8]}")

# Diccionario de datos (primera hoja)
# Lee todas las hojas del Excel; la primera tiene el diccionario de variables.
dic = pd.read_excel(os.path.join(BASE_PATH, "data", "docs", "Diccionario de datos_ECV 2023.xlsx"), sheet_name=None)
# Se queda con 4 columnas: módulo, nombre de la variable, texto de la pregunta y categorías de respuesta.
dd = dic[list(dic.keys())[0]].iloc[:, [0, 3, 9, 10]].copy()
dd.columns = ["modulo", "variable", "pregunta", "categorias"]
dd = dd.dropna(subset=["variable"]).reset_index(drop=True)

# buscar_dic('palabra'): lista las variables cuya pregunta o categorías contienen la palabra.
def buscar_dic(palabra, n=10):
    """Busca una palabra en las preguntas y categorías del diccionario."""
    pal = palabra.lower()
    m = (dd["pregunta"].astype(str).str.lower().str.contains(pal, regex=False)
         | dd["categorias"].astype(str).str.lower().str.contains(pal, regex=False))
    print(f"\n[{palabra}]")
    for _, f in dd[m].head(n).iterrows():
        print(f"  {f['variable']:10s} | {str(f['modulo'])[:24]:24s} | {str(f['pregunta'])[:95]}")

# ver('P6051'): imprime la pregunta y los códigos de respuesta de una variable.
def ver(var):
    """Muestra pregunta y categorías de una variable."""
    f = dd[dd["variable"] == var].iloc[0]
    print(f"{var} ({f['modulo']})\n{f['pregunta']}\n{f['categorias']}\n")

# Verificación del jefe de hogar: cada hogar debe tener exactamente un jefe (P6051 == 1).
ver("P6051")
jefes_chk = df_personas[df_personas["P6051"] == 1].groupby(LLAVE_HOGAR).size()
print("Hogares con exactamente 1 jefe:", (jefes_chk == 1).sum(), "de", df_hogares.shape[0])

**Conclusión.** El parentesco es `P6051` (1 = jefe/a del hogar) y el tamaño del hogar también viene en `CANT_PERSONAS_HOGAR`. `ESTRATO2020` es un código de diseño muestral, **no** el estrato socioeconómico; el estrato (1 a 6) es `P8520S1A1` en `vivienda`.

## 3. LIMPIEZA Y VARIABLES DERIVADAS

In [ ]:
# 3. LIMPIEZA Y VARIABLES DERIVADAS
# Qué hace: crea las variables que no vienen en los microdatos (años de educación, ocupado, sexo y edad del jefe, estrato válido) y revisa el ingreso.
# 'p' es un alias de df_personas: lo que se agregue aquí queda en df_personas.
p = df_personas

# 3.1 Años de educación aprobados (persona)
# Dos casos según si la persona estudia actualmente:
# - No estudiante: se usa el último nivel (P8587) y el grado aprobado (P8587S1).
# - Estudiante: años aprobados = grado que cursa - 1 (el grado en curso aún no está aprobado).
def anos_no_estudiante(n, g):
    g = g.fillna(0)
    return np.select(
        [n.isin([1, 2]), n == 3, n == 4, n == 5, n >= 6],
        [0, g.clip(0, 5), g.clip(5, 9), g.clip(9, 11), 11 + g],
        default=np.nan)

def anos_estudiante(n, g, sup):
    """Años aprobados = grado que cursa - 1 (metodología DANE)."""
    g = g.fillna(1)
    return np.select(
        [n == 1, n == 2, n == 3, n == 4, n >= 5],
        [0,
         (g - 1).clip(0, 4),
         (np.where(g >= 6, g, 5 + g) - 1).clip(5, 8),
         (np.where(g >= 10, g, 9 + g) - 1).clip(9, 11),
         11 + sup.fillna(0)],
        default=np.nan)

# Estudiante = declara que estudia (P8586 == 1) y reporta el nivel en que está matriculado (P1088).
estudia = (p["P8586"] == 1) & p["P1088"].notna()
# Combina los dos casos en una sola variable ANOS_EDUC.
p["ANOS_EDUC"] = np.where(
    estudia,
    anos_estudiante(p["P1088"], p["P1088S1"], p["P6216"]),
    anos_no_estudiante(p["P8587"], p["P8587S1"]))
p["ANOS_EDUC"] = p["ANOS_EDUC"].clip(upper=22)   # 11 años de básica y media + máximo 11 de superior
# OCUPADO: trabajó la semana pasada, o hizo una actividad paga, o tenía trabajo, o ayudó sin pago en un negocio familiar (P6240, P6250, P6260, P6270).
p["OCUPADO"] = ((p["P6240"] == 1) | (p["P6250"] == 1) | (p["P6260"] == 1) | (p["P6270"] == 1))

# Controles: personas de 15+ sin dato de educación (debe ser 0) y distribución de los años de educación.
m15 = p["P6040"] >= 15
print("Estudiantes con nivel matriculado:", estudia.sum())
print(p[estudia].groupby("P1088")["P1088S1"].agg(["min", "max", "count"]))
print("Personas 15+ sin dato de educación:", p.loc[m15, "ANOS_EDUC"].isna().sum())
print(p.loc[m15, "ANOS_EDUC"].describe())

# 3.2 Jefe de hogar, sexo y edad del jefe
# Se eliminan las columnas si ya existían (permite re-ejecutar la celda sin duplicarlas).
df_hogares = df_hogares.drop(columns=["SEXO_JEFE", "EDAD_JEFE"], errors="ignore")
# Jefe/a del hogar = persona con P6051 == 1; se toma su sexo (P6020) y su edad (P6040).
jefes = (p[p["P6051"] == 1][LLAVE_HOGAR + ["P6020", "P6040"]]
         .drop_duplicates(LLAVE_HOGAR)
         .rename(columns={"P6020": "SEXO_JEFE", "P6040": "EDAD_JEFE"}))
# Se agrega al hogar el sexo y la edad del jefe.
df_hogares = df_hogares.merge(jefes, on=LLAVE_HOGAR, how="left")
print("Hogares sin jefe:", df_hogares["SEXO_JEFE"].isna().sum())

# 3.3 Estrato (vivienda: "Estrato para tarifa"); solo valores 1 a 6 son estratos válidos
# Solo los valores 1 a 6 son estratos válidos; los códigos 0, 8, 9 y los vacíos quedan como nulos.
df_hogares["ESTRATO"] = df_hogares["P8520S1A1"].where(df_hogares["P8520S1A1"].between(1, 6))
print(df_hogares["ESTRATO"].value_counts(dropna=False).sort_index())

# 3.4 Ingreso del hogar
# Revisión del ingreso: distribución y porcentaje de hogares con ingreso 0 (se conservan en la base).
print(df_hogares[["I_HOGAR", "PERCAPITA"]].describe(percentiles=[.01, .25, .5, .75, .99]))
print("Hogares con I_HOGAR == 0:", round((df_hogares["I_HOGAR"] == 0).mean() * 100, 2), "%")

**Decisiones de limpieza**
- **Años de educación:** quien estudia cuenta los años hasta el grado anterior al que cursa (metodología DANE); el resto usa el último nivel y grado aprobados. Se recorta en 22 años (11 de básica y media + 11 de superior) porque `P6216` traía valores atípicos hasta 31.
- **Jefe de hogar:** `P6051 == 1`. Todos los hogares tienen jefe.
- **Estrato:** solo se aceptan valores de 1 a 6; los códigos 0, 8 y 9 y los vacíos quedan como nulos (11.019 hogares).
- **Ingreso:** el 1,9% de los hogares reporta ingreso 0 y hay valores extremos (máximo de 250 millones por persona). Se conservan en la base; el univariado excluye los ceros y usa escala logarítmica.
- **Códigos especiales:** en `P6090` (afiliación a salud) el código 9 ("no sabe") se trata como no asegurado, como indica la metodología del IPM.

## 4. VARIABLE OBJETIVO: ÍNDICE DE POBREZA MULTIDIMENSIONAL (IPM)
Los microdatos no traen el IPM, así que se reconstruye con la metodología del DANE: 15 indicadores en 5 dimensiones, cada dimensión pesa 0,2 y los indicadores de una misma dimensión pesan igual. Un hogar es pobre si su suma ponderada de privaciones es **≥ 1/3**.

| Dimensión | Indicador | Peso | Variables principales |
|---|---|---|---|
| Educación | I01 Bajo logro educativo (promedio < 9 años, 15+) | 0,10 | `P8587`, `P8587S1`, `P1088`, `P1088S1` |
| Educación | I02 Analfabetismo (15+) | 0,10 | `P6160` |
| Niñez y juventud | I03 Inasistencia escolar (6–16) | 0,05 | `P8586` |
| Niñez y juventud | I04 Rezago escolar (7–17) | 0,05 | `ANOS_EDUC`, `P6040` |
| Niñez y juventud | I05 Barreras de cuidado de primera infancia (0–5) | 0,05 | `P51`, `P6090`, `P8586` |
| Niñez y juventud | I06 Trabajo infantil (12–17) | 0,05 | `P6240`, `P6250`, `P6260`, `P6270` |
| Trabajo | I07 Desempleo de larga duración | 0,10 | `P7250` |
| Trabajo | I08 Empleo informal (sin pensión) | 0,10 | `P6920` |
| Salud | I09 Sin aseguramiento (6+) | 0,10 | `P6090` |
| Salud | I10 Barreras de acceso a salud | 0,10 | `P5665`, `P8563` |
| Vivienda | I11 Sin fuente de agua mejorada | 0,04 | `P8520S5`, `P8530` |
| Vivienda | I12 Inadecuada eliminación de excretas | 0,04 | `P8520S3`, `P8526` |
| Vivienda | I13 Pisos inadecuados | 0,04 | `P4015` |
| Vivienda | I14 Paredes inadecuadas | 0,04 | `P4005` |
| Vivienda | I15 Hacinamiento crítico | 0,04 | `P5010`, `TAM_HOGAR` |

Zona urbana = `CLASE` 1 (cabeceras); rural = `CLASE` 2 (centros poblados y rural disperso).

In [ ]:
# 4. CONSTRUCCIÓN DEL IPM (metodología DANE)
# Qué hace: calcula los 15 indicadores de privación por hogar (1 = privado, 0 = no privado), los pondera y define POBRE_IPM.
# Regla general: un indicador se activa si al menos una persona del hogar cumple la condición (en el grupo de edad que corresponde).
# Resultado: la tabla 'ind' (una fila por hogar) con I01..I15, SCORE_IPM y POBRE_IPM, que se une a df_hogares.
# Alias: p = base de personas, h = base de hogares, edad = edad de cada persona.
p = df_personas
h = df_hogares
edad = p["P6040"]
# 'ind' es la tabla de indicadores: una fila por hogar, indexada por la llave de hogar.
ind = h[LLAVE_HOGAR].copy().set_index(LLAVE_HOGAR)

# cuenta(mask): número de personas de cada hogar que cumplen una condición (mask es una condición sobre personas).
def cuenta(mask):
    """Personas por hogar que cumplen la condición."""
    return p[mask].groupby(LLAVE_HOGAR).size().reindex(ind.index, fill_value=0)

# DIMENSIÓN EDUCACIÓN
# I01 Bajo logro educativo: promedio de años de educación de las personas de 15+ del hogar menor a 9 (sin personas de 15+ = privado).
# I02 Analfabetismo: al menos una persona de 15+ que no sabe leer ni escribir (P6160 == 2).
m15 = edad >= 15
n15 = cuenta(m15)
prom = p[m15].groupby(LLAVE_HOGAR)["ANOS_EDUC"].mean().reindex(ind.index)
ind["I01_logro_educ"]    = ((prom < 9) | prom.isna()).astype(int)
ind["I02_analfabetismo"] = ((cuenta(m15 & (p["P6160"] == 2)) > 0) | (n15 == 0)).astype(int)

# DIMENSIÓN NIÑEZ Y JUVENTUD
# I03 Inasistencia escolar: al menos un niño de 6 a 16 años que no estudia (P8586 == 2).
# I04 Rezago escolar: al menos una persona de 7 a 17 años con menos años de educación que la norma (edad - 6).
ind["I03_inasistencia"] = (cuenta(edad.between(6, 16) & (p["P8586"] == 2)) > 0).astype(int)
ind["I04_rezago"]       = (cuenta(edad.between(7, 17) & (p["ANOS_EDUC"] < (edad - 6))) > 0).astype(int)

# I05 Primera infancia (0 a 5 años): privado si no está afiliado a salud, o si permanece con el padre/madre en el trabajo,
# con un pariente menor de 18 años o solo (P51 = 3, 6, 7). A los 5 años: sin afiliación o sin estudiar.
# (No incluye el componente de alimentación en el jardín.)
sin_salud = p["P6090"] != 1                      # 2 = no, 9 = no sabe (se cuenta como no asegurado)
priv_0a4  = edad.between(0, 4) & (p["P51"].isin([3, 6, 7]) | sin_salud)
priv_5    = (edad == 5) & (sin_salud | (p["P8586"] == 2))
ind["I05_primera_infancia"] = (cuenta(priv_0a4 | priv_5) > 0).astype(int)
# I06 Trabajo infantil: al menos una persona de 12 a 17 años ocupada.
ind["I06_trabajo_infantil"] = (cuenta(edad.between(12, 17) & p["OCUPADO"]) > 0).astype(int)

# DIMENSIÓN TRABAJO
# Desocupado = no ocupado pero con tiempo de búsqueda de trabajo reportado (P7250). PEA = ocupados + desocupados.
# I07 Desempleo de larga duración: alguien busca trabajo hace más de 52 semanas; los hogares sin PEA se consideran privados (regla DANE).
desoc = ~p["OCUPADO"] & p["P7250"].notna()
pea   = p["OCUPADO"] | desoc
dld   = desoc & (p["P7250"] > 52)                # buscando trabajo más de 12 meses
ind["I07_desempleo_larga"] = ((cuenta(dld) > 0) | (cuenta(pea) == 0)).astype(int)

# I08 Informalidad: hogar con algún miembro de la PEA (sin contar menores de 18 ocupados ni desempleados de larga duración) que no cotiza a pensión.
# Interruptor: si es True, los hogares sin ocupados y sin pensionados también quedan privados.
SIN_PEA_PRIVADO_INFORMAL = True                  # si la incidencia sale muy alta, probar False
pea2   = (pea & ~dld) & ~(p["OCUPADO"] & (edad < 18))
formal = p["OCUPADO"] & p["P6920"].isin([1, 3])
n_pea2, n_formal = cuenta(pea2), cuenta(pea2 & formal)
pensionado = cuenta(p["P6920"] == 3) > 0
sin_pea2 = (n_pea2 == 0) & (~pensionado if SIN_PEA_PRIVADO_INFORMAL else False)
ind["I08_informal"] = (((n_pea2 > 0) & (n_formal != n_pea2)) | sin_pea2).astype(int)

# DIMENSIÓN SALUD
# I09 Sin aseguramiento: al menos una persona de 6+ años sin afiliación a salud (el código 9 'no sabe' cuenta como no asegurado).
# I10 Barreras de acceso: tuvo un problema de salud en los últimos 30 días (P5665 == 1) y no acudió a su entidad de salud ni a un profesional (P8563 distinto de 1 y 2).
ind["I09_sin_aseguramiento"] = (cuenta((edad >= 6) & sin_salud) > 0).astype(int)
ind["I10_barreras_salud"]    = (cuenta((p["P5665"] == 1) & ~p["P8563"].isin([1, 2])) > 0).astype(int)

# DIMENSIÓN VIVIENDA Y SERVICIOS (los criterios cambian entre zona urbana y rural)
# I11 Agua: urbano = sin acueducto; rural = agua de pozo sin bomba, lluvia, río, carro tanque, aguatero o embotellada.
# I12 Excretas: urbano = sin alcantarillado; rural = inodoro sin conexión, bajamar o sin servicio sanitario.
# I13 Pisos: de tierra. I14 Paredes: materiales precarios (en zona urbana incluye madera burda).
# I15 Hacinamiento: urbano = 3 o más personas por cuarto para dormir; rural = más de 3.
AGUA_RURAL_PRIV = [4, 5, 6, 8, 9, 10]            # pozo sin bomba, lluvia, río, carro tanque, aguatero, embotellada
urb = h["CLASE"] == 1
ppc = h["TAM_HOGAR"] / h["P5010"].clip(lower=1)  # personas por cuarto para dormir
ind["I11_agua"]         = np.where(urb, h["P8520S5"] != 1, h["P8530"].isin(AGUA_RURAL_PRIV)).astype(int)
ind["I12_excretas"]     = np.where(urb, h["P8520S3"] != 1, h["P8526"].isin([3, 5, 6])).astype(int)
ind["I13_pisos"]        = (h["P4015"] == 7).astype(int).values
ind["I14_paredes"]      = np.where(urb, h["P4005"].isin([5, 7, 8, 9, 10]), h["P4005"].isin([7, 8, 9, 10])).astype(int)
ind["I15_hacinamiento"] = np.where(urb, ppc >= 3, ppc > 3).astype(int)

# PONDERACIÓN: cada dimensión pesa 0,2 (educación, trabajo y salud: 0,10 por indicador; niñez: 0,05; vivienda: 0,04).
PESOS = {"I01_logro_educ": .10, "I02_analfabetismo": .10,
         "I03_inasistencia": .05, "I04_rezago": .05, "I05_primera_infancia": .05, "I06_trabajo_infantil": .05,
         "I07_desempleo_larga": .10, "I08_informal": .10,
         "I09_sin_aseguramiento": .10, "I10_barreras_salud": .10,
         "I11_agua": .04, "I12_excretas": .04, "I13_pisos": .04, "I14_paredes": .04, "I15_hacinamiento": .04}
COLS_IND = list(PESOS)
assert abs(sum(PESOS.values()) - 1) < 1e-9

# SCORE_IPM = suma ponderada de privaciones (0 a 1). POBRE_IPM = 1 si el hogar llega a 1/3 (33,3%) de privación ponderada.
ind["SCORE_IPM"] = sum(ind[c] * w for c, w in PESOS.items())
ind["POBRE_IPM"] = (ind["SCORE_IPM"] >= 1/3 - 1e-9).astype(int)

# Se agregan los indicadores y el puntaje a df_hogares (el drop previo permite re-ejecutar sin duplicar columnas).
df_hogares = df_hogares.drop(columns=COLS_IND + ["SCORE_IPM", "POBRE_IPM"], errors="ignore")
df_hogares = df_hogares.merge(ind.reset_index(), on=LLAVE_HOGAR, how="left")
# Control: no se perdieron hogares.
assert len(df_hogares) == 86405

### 4.1 Validación contra las cifras oficiales del DANE 2023

In [ ]:
# 4.1 VALIDACIÓN CONTRA EL DANE 2023
# Qué hace: calcula la incidencia de pobreza multidimensional a nivel de PERSONAS (así la publica el DANE), ponderada con FEX_C, y la compara con las cifras oficiales.
# También compara la intensidad (privación promedio de los pobres) y el IPM ajustado M0 = H x A.
# Cada persona hereda el estado de pobreza de su hogar (unión por la llave de hogar).
pp = (df_personas[LLAVE_HOGAR + ["FEX_C"]]
      .merge(df_hogares[LLAVE_HOGAR + ["POBRE_IPM", "SCORE_IPM", "CLASE", "SEXO_JEFE"]], on=LLAVE_HOGAR))
# H(d): incidencia ponderada en % (promedio de POBRE_IPM con pesos FEX_C).
H = lambda d: 100 * np.average(d["POBRE_IPM"], weights=d["FEX_C"])

# Tabla comparativa: cifras propias vs. cifras oficiales del DANE (boletín técnico, datos 2023).
val = pd.DataFrame({
    "propio": [H(pp), H(pp[pp.CLASE == 1]), H(pp[pp.CLASE == 2]),
               H(pp[pp.SEXO_JEFE == 1]), H(pp[pp.SEXO_JEFE == 2])],
    "DANE_2023": [12.1, 8.3, 25.1, 11.2, 13.2]},
    index=["Nacional", "Cabeceras", "CP y rural disperso", "Jefe hombre", "Jefe mujer"])
print(val.round(1))

# Intensidad A: promedio del puntaje de privación entre las personas pobres.
pobres = pp[pp.POBRE_IPM == 1]
A = 100 * np.average(pobres["SCORE_IPM"], weights=pobres["FEX_C"])
print(f"\nIntensidad A: {A:.1f}%  (DANE 40.1%)   |   M0 = H*A: {H(pp)/100*A/100:.3f}  (DANE 0.049)")

# Privación por indicador (% de hogares, ponderado)
w = df_hogares["FEX_C"]
print((df_hogares[COLS_IND].apply(lambda c: 100 * np.average(c, weights=w))).round(1))

**Resultado de la validación** (% de **personas**, como lo publica el DANE; cifras oficiales del boletín técnico de pobreza multidimensional):

| | Propio | DANE 2023 |
|---|---|---|
| Nacional | 11,7% | 12,1% |
| Cabeceras | 7,6% | 8,3% |
| Centros poblados y rural disperso | 24,8% | 25,1% |
| Jefe hombre | 11,0% | 11,2% |
| Jefe mujer | 12,5% | 13,2% |
| Intensidad (A) | 40,2% | 40,1% |
| IPM ajustado (M0) | 0,047 | 0,049 |

La reconstrucción queda a menos de 1 punto porcentual del dato oficial y la intensidad coincide. Las diferencias se explican, probablemente, porque (i) el indicador de primera infancia no incluye el componente de alimentación en el jardín, (ii) el DANE excluye la ruralidad de Amazonía–Orinoquía en sus totales y (iii) hay decisiones propias sobre hogares sin ocupados y pensionados. El indicador de bajo logro educativo sale más alto de lo esperado (≈ 40%) y queda como limitación.

### 4.2 Guardar bases limpias

In [ ]:
# 4.2 GUARDAR LAS BASES LIMPIAS
# Qué hace: guarda en data/processed/ solo las columnas que se usan (hogares y personas) en formato parquet.
# Por qué: la Sección 6 y cualquier análisis posterior cargan estos archivos y no necesitan repetir la limpieza.
COLS_HOGAR = (LLAVE_HOGAR + ["FEX_C", "CLASE", "REGION", "P1_DEPARTAMENTO", "TAM_HOGAR",
              "I_HOGAR", "PERCAPITA", "ESTRATO", "SEXO_JEFE", "EDAD_JEFE"]
              + COLS_IND + ["SCORE_IPM", "POBRE_IPM"])
COLS_PERS = LLAVE_PERSONA + ["FEX_C", "P6040", "P6020", "P6051", "P6160", "P8586",
                             "P8587", "P8587S1", "ANOS_EDUC", "OCUPADO", "P6090"]

# Rutas de salida (parquet pesa unos 2 MB por archivo, mucho menos que CSV).
ruta_h = os.path.join(PROCESSED, "ecv2023_hogares_limpia.parquet")
ruta_p = os.path.join(PROCESSED, "ecv2023_personas_limpia.parquet")
# Escritura de las dos bases.
df_hogares[COLS_HOGAR].to_parquet(ruta_h, index=False)
df_personas[COLS_PERS].to_parquet(ruta_p, index=False)

for r in (ruta_h, ruta_p):
    print(os.path.basename(r), round(os.path.getsize(r) / 1e6, 1), "MB")

## 5. ANÁLISIS UNIVARIADO
*Estadísticas de la muestra, sin factor de expansión.*

In [ ]:
# 5. ANÁLISIS UNIVARIADO
# Qué hace: resume cada variable numérica clave con medidas de tendencia central, dispersión y forma.
# Variables: ingreso per cápita del hogar, edad, años de educación (15+) y tamaño del hogar. Son estadísticas de la muestra, sin factor de expansión.
# Alias de las bases integradas.
hog, per = df_hogares, df_personas
# Variables a analizar: ingreso per cápita (sin los hogares con ingreso 0) y años de educación de personas de 15+.
ing  = hog.loc[hog["PERCAPITA"] > 0, "PERCAPITA"]          # excluye 1,9% de hogares con ingreso 0
educ = per.loc[per["P6040"] >= 15, "ANOS_EDUC"]

# resumen_uni: n, media, mediana, desviación estándar, mínimo, Q1, Q3, IQR, máximo, asimetría y curtosis.
def resumen_uni(s, nombre):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({"n": len(s), "media": s.mean(), "mediana": s.median(), "desv_est": s.std(),
                      "min": s.min(), "Q1": q1, "Q3": q3, "IQR": q3 - q1, "max": s.max(),
                      "asimetria": s.skew(), "curtosis": s.kurt()}, name=nombre)

# Tabla univariada (el ingreso se resume también en log10 porque está muy sesgado a la derecha).
tabla_uni = pd.concat([
    resumen_uni(ing, "Ingreso per cápita (COP)"),
    resumen_uni(np.log10(ing), "log10 ingreso per cápita"),
    resumen_uni(per["P6040"], "Edad (años)"),
    resumen_uni(educ, "Años de educación (15+)"),
    resumen_uni(hog["TAM_HOGAR"], "Tamaño del hogar"),
], axis=1).T
display(tabla_uni)

# Porcentaje de hogares con ingreso atípico alto según la regla de 1,5 x IQR.
q1, q3 = ing.quantile([.25, .75]); iqr = q3 - q1
print("Outliers altos de ingreso (regla 1,5·IQR):", round((ing > q3 + 1.5 * iqr).mean() * 100, 1), "%")

In [ ]:
# 5.1 HISTOGRAMAS Y 5.2 BOXPLOTS
# Qué hace: grafica la distribución de las cuatro variables (histogramas) y sus valores atípicos (boxplots).
# Guarda las figuras en figures/ para usarlas en el informe.
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(np.log10(ing), bins=50, ax=ax[0, 0]); ax[0, 0].set(title="Ingreso per cápita del hogar (log10 COP)", xlabel="log10(COP)")
sns.histplot(per["P6040"], bins=range(0, 105, 5), ax=ax[0, 1]); ax[0, 1].set(title="Edad", xlabel="años")
sns.histplot(educ.clip(upper=22), bins=range(0, 24), ax=ax[1, 0]); ax[1, 0].set(title="Años de educación (personas de 15+)", xlabel="años")
sns.countplot(x=hog["TAM_HOGAR"].clip(upper=8), ax=ax[1, 1]); ax[1, 1].set(title="Tamaño del hogar (8 = 8 o más)", xlabel="personas")
plt.tight_layout(); plt.savefig(os.path.join(FIGURES, "05_histogramas.png"), dpi=150); plt.show()

# 5.2 Boxplots
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
sns.boxplot(y=np.log10(ing), ax=ax[0]); ax[0].set(title="log10 ingreso per cápita", ylabel="")
sns.boxplot(y=per["P6040"], ax=ax[1]); ax[1].set(title="Edad", ylabel="")
sns.boxplot(y=educ, ax=ax[2]); ax[2].set(title="Años de educación (15+)", ylabel="")
sns.boxplot(y=hog["TAM_HOGAR"], ax=ax[3]); ax[3].set(title="Tamaño del hogar", ylabel="")
plt.tight_layout(); plt.savefig(os.path.join(FIGURES, "05_boxplots.png"), dpi=150); plt.show()

### Conclusiones del análisis univariado
*Estadísticas de la muestra, sin factor de expansión. Las cifras provienen de la tabla `tabla_uni`; verificarlas al volver a ejecutar.*

**Ingreso per cápita del hogar** (n = 84.768; se excluyó el 1,9% de hogares con ingreso 0). La media (≈ \$940.000) es cerca de 1,8 veces la mediana (≈ \$518.000), con desviación estándar de ≈ \$2,06 millones, asimetría de ≈ 42 y curtosis de ≈ 4.000: distribución extremadamente sesgada a la derecha, con valores extremos de hasta \$250 millones. El IQR es de \$712.500 (Q1 \$287.500; Q3 \$1.000.000). El 8,4% de los hogares supera el límite de atípicos de la regla 1,5·IQR (Q3 + 1,5·IQR ≈ \$2,07 millones). En escala log10 la distribución es casi simétrica (media 5,72; mediana 5,71; asimetría -0,17), lo que justifica usar el logaritmo en gráficos y modelos. El boxplot muestra además una cola baja de ingresos muy pequeños (mínimo de ≈ \$21 por persona), que se conserva y se declara como limitación.

**Edad** (n = 240.212). Media 34,7 y mediana 33 años, con rango de 0 a 104. IQR de 36 años (Q1 16; Q3 52), asimetría de 0,35 y curtosis de -0,83. Es una población joven: la mayor concentración está entre los 0 y 20 años, con cola decreciente hacia edades avanzadas.

**Años de educación, personas de 15 años o más** (n = 184.563; recortado en 22 años). Media ≈ 8,1 y mediana 9 años (básica secundaria). El 25% tiene 5 años o menos y el 75% tiene 11 años o menos. Distribución casi simétrica y aplanada, con picos en 5 y 11 años, que corresponden a primaria y bachillerato completos.

**Tamaño del hogar** (n = 86.405). Media 2,78 y mediana 3 personas, con moda en 2. IQR de 2 (Q1 2; Q3 4), asimetría de 1,10 y curtosis de 2,51: la mayoría de los hogares tiene entre 2 y 4 personas y los hogares grandes son una minoría (máximo 24).

## Estado y pendientes
**Listo:** secciones 0 a 8. Bases limpias en `data/processed/` (hogares y personas, en parquet) y figuras en `figures/`.

**IPM:** reconstrucción propia validada contra el DANE 2023. Incidencia de **personas**: 11,7% (oficial 12,1%); intensidad 40,2% (oficial 40,1%). A nivel de **hogares** la incidencia es 9,54%.

**Limitaciones:** ver la Sección 8.

**Pendiente:** modelo de regresión logística y pruebas de hipótesis (ver *Siguientes pasos*, Sección 8).

## 6. ANÁLISIS MULTIVARIADO

In [ ]:
# CARGA DE LAS BASES LIMPIAS (Sección 6)
# Qué hace: carga los parquet generados en 4.2 para trabajar sin repetir la limpieza.
# Nota: PESOS se repite aquí (iguales a los de la Sección 4) para poder ejecutar la Sección 6 de forma independiente.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas: raíz del repo y carpetas de datos procesados y figuras.
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED = os.path.join(BASE_PATH, "data", "processed")
FIGURES   = os.path.join(BASE_PATH, "figures")
sns.set_theme(style="whitegrid", context="notebook")

# hog: una fila por hogar (indicadores I01-I15, SCORE_IPM, POBRE_IPM, FEX_C...). per: una fila por persona.
hog = pd.read_parquet(os.path.join(PROCESSED, "ecv2023_hogares_limpia.parquet"))
per = pd.read_parquet(os.path.join(PROCESSED, "ecv2023_personas_limpia.parquet"))
# Pesos oficiales de los indicadores (los mismos de la Sección 4).
PESOS = {"I01_logro_educ": .10, "I02_analfabetismo": .10,
         "I03_inasistencia": .05, "I04_rezago": .05, "I05_primera_infancia": .05, "I06_trabajo_infantil": .05,
         "I07_desempleo_larga": .10, "I08_informal": .10,
         "I09_sin_aseguramiento": .10, "I10_barreras_salud": .10,
         "I11_agua": .04, "I12_excretas": .04, "I13_pisos": .04, "I14_paredes": .04, "I15_hacinamiento": .04}

print(hog.shape, per.shape)
hog.head()

### 6.1 Correlaciones con el puntaje de pobreza

In [ ]:
# 6.1 CORRELACIONES
# Qué hace: mide la correlación de Pearson entre el puntaje de pobreza (SCORE_IPM) y tres variables numéricas del hogar:
# ingreso per cápita (log10), tamaño del hogar y edad del jefe.
# Resultado: la correlación más fuerte es con el ingreso (-0,51); tamaño del hogar y edad del jefe se relacionan débilmente (0,15 y 0,17).
NUM_VARS = ["SCORE_IPM", "PERCAPITA", "TAM_HOGAR", "EDAD_JEFE"]

# Igual que en el univariado, el ingreso per cápita se ve mejor en log10
hog["LOG_PERCAPITA"] = np.where(hog["PERCAPITA"] > 0, np.log10(hog["PERCAPITA"]), np.nan)
NUM_VARS_LOG = ["SCORE_IPM", "LOG_PERCAPITA", "TAM_HOGAR", "EDAD_JEFE"]

# Matriz de correlaciones y su mapa de calor.
corr = hog[NUM_VARS_LOG].corr(method="pearson")
display(corr.round(3))

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlación entre variables numéricas (hogares)")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, "06_heatmap_correlaciones.png"), dpi=150)
plt.show()

### 6.2 Privación por indicador (% de hogares)

In [ ]:
# 6.2 PRIVACIÓN POR INDICADOR
# Qué hace: calcula el % de hogares privados en cada uno de los 15 indicadores, ponderado con FEX_C (representa a toda la población).
# Resultado: la informalidad (I08) es la privación más frecuente (69,7%), seguida del bajo logro educativo (39,8%).
# Lista de los 15 indicadores.
indicadores = ["I01_logro_educ", "I02_analfabetismo", "I03_inasistencia", "I04_rezago",
               "I05_primera_infancia", "I06_trabajo_infantil", "I07_desempleo_larga",
               "I08_informal", "I09_sin_aseguramiento", "I10_barreras_salud",
               "I11_agua", "I12_excretas", "I13_pisos", "I14_paredes", "I15_hacinamiento"]

# Tasa ponderada = suma(privación x FEX_C) / suma(FEX_C) x 100.
def tasa_privacion_ponderada(df, col, peso="FEX_C"):
    return (df[col] * df[peso]).sum() / df[peso].sum() * 100

resumen_indicadores = pd.DataFrame({
    "indicador": indicadores,
    "pct_privacion_ponderado": [tasa_privacion_ponderada(hog, i) for i in indicadores]
}).sort_values("pct_privacion_ponderado", ascending=False)

display(resumen_indicadores)

plt.figure(figsize=(8, 6))
sns.barplot(data=resumen_indicadores, y="indicador", x="pct_privacion_ponderado", color="#4C72B0")
plt.xlabel("% de hogares privados (ponderado por FEX_C)")
plt.title("Privación por indicador del IPM")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, "06_barras_indicadores_ipm.png"), dpi=150)
plt.show()

### 6.3 Incidencia por zona, región, sexo del jefe y estrato (% de hogares)

In [ ]:
# 6.3 INCIDENCIA DE POBREZA POR GRUPOS (% de hogares)
# Qué hace: calcula el % de hogares pobres (POBRE_IPM = 1), ponderado con FEX_C, por zona (CLASE), región, sexo del jefe y estrato, y grafica cada comparación.
# Resultado: la incidencia varía mucho por zona (rural 22,4% vs urbano 5,8%) y región, y poco por sexo del jefe (10,2% vs 9,0%).
def incidencia_ponderada(df, grupo, peso="FEX_C", pobre="POBRE_IPM"):
    out = df.groupby(grupo).apply(
        lambda g: (g[pobre] * g[peso]).sum() / g[peso].sum() * 100
    ).reset_index(name="incidencia_pct")
    return out

# Repite el cálculo y el gráfico para cada variable de agrupación.
for grupo in ["CLASE", "REGION", "SEXO_JEFE", "ESTRATO"]:
    tabla = incidencia_ponderada(hog, grupo)
    print(f"\n--- Incidencia de pobreza multidimensional por {grupo} ---")
    display(tabla)

    plt.figure(figsize=(7, 4))
    sns.barplot(data=tabla, x=grupo, y="incidencia_pct", color="#55A868")
    plt.ylabel("Incidencia IPM (%)")
    plt.title(f"Incidencia de pobreza multidimensional por {grupo}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES, f"06_barras_incidencia_{grupo.lower()}.png"), dpi=150)
    plt.show()

REGION (agrupación de departamentos según la ficha técnica de la ECV, V16, p. 38):
1=Caribe, 2=Oriental, 3=Central, 4=Pacífica (Cauca, Chocó y Nariño, sin Valle), 5=Bogotá (cabecera), 6=Antioquia, 7=Valle del Cauca, 8=San Andrés, 9=Orinoquía-Amazonía

SEXO_JEFE:
1=Hombre, 2=Mujer

CLASE:
1=Urbano (cabecera municipal), 2=Rural (centros poblados y rural disperso)


In [ ]:
# 6.3b VERIFICACIÓN DEL MAPEO DE REGION
# Qué hace: compara el código REGION de cada departamento con la agrupación oficial de la ficha técnica de la ECV (V16, p. 38).
# Resultado: solo Bogotá (11) aparece en dos regiones (2 y 5), porque su zona rural pertenece a la región Oriental.
ESPERADA = {1: [8, 13, 20, 23, 44, 47, 70],         # Caribe
            2: [15, 25, 50, 54, 68],                 # Oriental
            3: [17, 18, 41, 63, 66, 73],             # Central
            4: [19, 27, 52],                         # Pacífica (sin Valle)
            5: [11], 6: [5], 7: [76], 8: [88],       # Bogotá, Antioquia, Valle, San Andrés
            9: [81, 85, 86, 91, 94, 95, 97, 99]}     # Orinoquía-Amazonía
esp = {d: r for r, ds in ESPERADA.items() for d in ds}
reg_datos = hog.groupby("P1_DEPARTAMENTO")["REGION"].agg(lambda s: sorted(s.unique()))
chk = pd.DataFrame({"en_datos": reg_datos, "esperada": pd.Series(esp)})
chk["coincide"] = [d == [e] for d, e in zip(chk["en_datos"], chk["esperada"])]
display(chk[~chk["coincide"]])    # vacío = todo coincide (Bogotá aparece por su zona rural)

**Conclusión.** Los 32 departamentos coinciden con la agrupación de la ficha técnica V16 (p. 38). La única excepción es Bogotá, cuya zona rural pertenece a la región Oriental; por eso la región 5 corresponde a Bogotá (cabecera).

### 6.4 Incidencia a nivel de personas

In [ ]:
# 6.4 INCIDENCIA A NIVEL DE PERSONAS
# Qué hace: asigna a cada persona el estado de pobreza de su hogar y calcula la incidencia ponderada con el FEX_C de personas.
# Por qué: el DANE publica la incidencia como % de personas; esta es la cifra comparable con el 12,1% oficial (aquí 11,66%).
personas_pobreza = per.merge(
    hog[["DIRECTORIO", "SECUENCIA_P", "POBRE_IPM", "CLASE"]],
    on=["DIRECTORIO", "SECUENCIA_P"], how="left"
)

incidencia_personas = (personas_pobreza["POBRE_IPM"] * personas_pobreza["FEX_C"]).sum() \
                       / personas_pobreza["FEX_C"].sum() * 100
print(f"Incidencia de pobreza multidimensional (personas, nacional): {incidencia_personas:.2f}%")

# Por CLASE, a nivel persona (para comparar con la validación de la Sección 4.1: 7,6% / 24,8%)
inc_persona_clase = personas_pobreza.groupby("CLASE").apply(
    lambda g: (g["POBRE_IPM"] * g["FEX_C"]).sum() / g["FEX_C"].sum() * 100
).reset_index(name="incidencia_pct")
display(inc_persona_clase)

### 6.5 Contribución por dimensión y comparación entre pobres y no pobres

In [ ]:
# 6.5 CONTRIBUCIÓN DE LAS DIMENSIONES ENTRE LOS HOGARES POBRES
# Qué hace: (1) agrupa los 15 indicadores en las 5 dimensiones y calcula cuánto aporta cada una a la privación ponderada de los hogares pobres (así lo mide el DANE);
# (2) compara el % de privación de pobres vs no pobres en cada indicador (razón = pobres / no pobres).
# Resultado: educación (37,4%) y trabajo (32,4%) aportan casi 70%; el analfabetismo separa mucho más a pobres de no pobres que la informalidad.
# Ojo: POBRE_IPM se construye con estos mismos indicadores, así que parte de la diferencia es mecánica.
DIMENSIONES = {
    "Educación": ["I01_logro_educ", "I02_analfabetismo"],
    "Niñez y juventud": ["I03_inasistencia", "I04_rezago", "I05_primera_infancia", "I06_trabajo_infantil"],
    "Trabajo": ["I07_desempleo_larga", "I08_informal"],
    "Salud": ["I09_sin_aseguramiento", "I10_barreras_salud"],
    "Vivienda y servicios": ["I11_agua", "I12_excretas", "I13_pisos", "I14_paredes", "I15_hacinamiento"],
}
# Hogares pobres y no pobres.
pob = hog[hog["POBRE_IPM"] == 1]
no  = hog[hog["POBRE_IPM"] == 0]

# Aporte de cada indicador = % de pobres privados x peso del indicador; luego se suma por dimensión y se expresa como % del total.
aporte = pd.Series({c: np.average(pob[c], weights=pob["FEX_C"]) * PESOS[c] for c in indicadores})
contrib = pd.Series({d: aporte[cols].sum() for d, cols in DIMENSIONES.items()})
resumen_dim = (contrib / aporte.sum() * 100).round(1).sort_values(ascending=False).rename("pct_contribucion_pobres").to_frame()
display(resumen_dim)

plt.figure(figsize=(7, 5))
sns.barplot(x=resumen_dim["pct_contribucion_pobres"], y=resumen_dim.index, color="#C44E52")
plt.xlabel("% de la privación ponderada de los hogares pobres")
plt.title("Contribución de cada dimensión del IPM entre los hogares pobres")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, "06_barras_dimensiones_ipm.png"), dpi=150)
plt.show()

# ¿Qué indicadores distinguen a los pobres de los no pobres?
cmp = pd.DataFrame({
    "pobres_%":    [100 * np.average(pob[c], weights=pob["FEX_C"]) for c in indicadores],
    "no_pobres_%": [100 * np.average(no[c],  weights=no["FEX_C"])  for c in indicadores]},
    index=indicadores)
cmp["razon"] = cmp["pobres_%"] / cmp["no_pobres_%"]
display(cmp.round(1).sort_values("razon", ascending=False))

# Incidencia nacional a nivel de HOGARES (distinta de la de personas, 11,66%).
print("Incidencia nacional, hogares:", round(100 * np.average(hog["POBRE_IPM"], weights=hog["FEX_C"]), 2), "%")

### Conclusiones del análisis multivariado

**Relación entre ingreso y pobreza.** Las variables numéricas muestran que el ingreso por persona del hogar es la que más se relaciona con la pobreza multidimensional: entre más alto el ingreso, más bajo el puntaje de pobreza (correlación de -0,51). El tamaño del hogar (0,15) y la edad del jefe de hogar (0,17) se relacionan débilmente con el puntaje de pobreza.

**El trabajo informal es la privación más común.** De los 15 indicadores que componen el índice, el trabajo informal es, por mucho, el más frecuente: casi 7 de cada 10 hogares (69,7%) están en esta situación (el indicador incluye también a los hogares sin ocupados que no son pensionados). Le sigue el bajo nivel educativo (39,8%). El resto de indicadores afecta a menos del 25% de los hogares.

**Grandes diferencias entre zonas.** La incidencia de pobreza multidimensional (% de hogares) es casi 4 veces más alta en zonas rurales (22,4%) que en zonas urbanas (5,8%), y varía muchísimo según la región: Bogotá (cabecera) tiene la incidencia más baja (2,4%), mientras que Orinoquía-Amazonía, el Caribe y la región Pacífica superan el 16%. El estrato socioeconómico muestra el mismo patrón: a menor estrato, mayor pobreza, bajando de 15,2% en estrato 1 a casi 0% en estrato 6.

**Diferencias por sexo del jefe de hogar.** Los hogares con jefatura de mujeres tienen una incidencia un poco más alta (10,2%) que los de jefatura de hombres (9,0%), aunque la diferencia es menor que la de zona o región.

**Qué pesa más entre los hogares pobres.** Al agrupar los 15 indicadores en las 5 dimensiones y calcular su aporte entre los hogares pobres, educación (37,4%) y trabajo (32,4%) concentran casi 70% de la privación ponderada; vivienda (13,1%), niñez (9,7%) y salud (7,4%) aportan el resto. La informalidad, aunque muy común (69,7%), distingue poco entre pobres (99,1%) y no pobres (66,6%), mientras que el analfabetismo (53,3% frente a 2,8%) los separa mucho más. Estas comparaciones son descriptivas: POBRE_IPM se construye con los mismos indicadores, así que parte de la diferencia es mecánica.

## 7. HALLAZGOS

### Hallazgo 1 — Educación y trabajo concentran la privación de los hogares pobres

Entre los hogares pobres, educación (37,4%) y trabajo (32,4%) concentran casi 70% de la privación ponderada; vivienda aporta 13,1%, niñez 9,7% y salud 7,4%. Casi todos los hogares pobres tienen bajo logro educativo (95,7%) e informalidad (99,1%).

**¿Por qué es relevante?** Muestra dónde se concentran las carencias de los hogares pobres: casi siete de cada diez puntos de privación vienen de educación y trabajo, y los dos primeros lugares coinciden con el DANE (35,9% y 30% en 2024). Apunta a que el análisis y las políticas contra la pobreza multidimensional deberían priorizar el capital educativo y la calidad del empleo, más allá de vivienda y salud. Es una descripción de asociación: no demuestra que mejorar esas dimensiones reduzca la pobreza.

### Hallazgo 2 — La incidencia varía mucho por zona y región, y poco por el sexo del jefe

La incidencia es 22,4% en hogares rurales frente a 5,8% en urbanos, y va de 2,4% en Bogotá (cabecera) a 20,2% en Orinoquía-Amazonía. Según el sexo del jefe la brecha es de solo 1,2 puntos (10,2% mujeres, 9,0% hombres).

**¿Por qué es relevante?** Confirma una fuerte dimensión territorial de la pobreza: los hogares rurales y las regiones de Orinoquía-Amazonía, Caribe y Pacífica concentran las mayores incidencias, mientras que el sexo del jefe marca una diferencia pequeña. Esto respalda diseñar estrategias diferenciadas por zona y región. La comparación es descriptiva y no controla otros factores, como ingreso, educación o estrato, que se evaluarán en el modelo.

### Hallazgo 3 — La informalidad es la privación más frecuente, pero distingue poco

La informalidad es la privación más frecuente (69,7% de los hogares), pero distingue poco: afecta al 99,1% de los pobres y al 66,6% de los no pobres. Analfabetismo separa más: 53,3% frente a 2,8%, 19 veces.

**¿Por qué es relevante?** Una privación frecuente no necesariamente identifica la pobreza: dos de cada tres hogares no pobres también presentan informalidad. Indicadores menos comunes, como analfabetismo, inasistencia escolar (13,7% frente a 1,1%) o pisos de tierra (27,9% frente a 3,0%), separan mucho mejor a los hogares pobres. Parte de esa diferencia es mecánica, porque POBRE_IPM se construye con esos indicadores. Por eso el modelo de la siguiente etapa usará variables externas al índice, no los indicadores.

## 8. CONCLUSIONES Y SIGUIENTES PASOS

### Conclusión general

El análisis descriptivo y multivariado de la ECV 2023 muestra que, entre los hogares pobres, la privación se concentra en educación (37,4%) y trabajo (32,4%), y que la incidencia varía mucho más por zona y región que por el sexo del jefe de hogar. La incidencia reconstruida es 9,54% de los hogares y 11,66% de las personas; esta última queda cerca de la cifra oficial del DANE (12,1%, publicada como % de personas), lo que respalda la calidad de la reconstrucción del índice.

### Limitaciones conocidas

- El indicador de bajo logro educativo (I01) sale más alto de lo esperado (≈40%), posiblemente por decisiones propias en la reconstrucción del IPM (ver sección 4.1).
- El indicador de primera infancia (I05) no incluye el componente de alimentación en el jardín, por no estar disponible en los microdatos.
- 11.019 hogares quedaron sin estrato válido y no entran en esa comparación.
- El análisis univariado (sección 5) no usó el factor de expansión; el multivariado (sección 6) sí lo usa en todas las comparaciones.
- Los resultados de "incidencia por hogar" e "incidencia por persona" no son el mismo número (por ejemplo, zona rural: 22,4% vs. 24,9%) porque los hogares pobres tienden a ser más grandes; ambos son correctos, pero miden cosas distintas.
- Las diferencias entre hogares pobres y no pobres son en parte mecánicas, porque POBRE_IPM se construye con los mismos indicadores.

### Siguientes pasos

1. **Modelo de regresión logística para `POBRE_IPM`.** Como el IPM se calcula con los 15 indicadores, usarlos como variables explicativas sería circular (el modelo acertaría por construcción). El modelo debe usar variables externas al índice: ingreso per cápita (log10), estrato, zona, región, sexo y edad del jefe, y tamaño del hogar.
2. **Pruebas de hipótesis** sobre las diferencias observadas: chi-cuadrado de `POBRE_IPM` frente a zona, región y sexo del jefe, y ANOVA (o Kruskal-Wallis) del ingreso entre grupos. Con una muestra compleja, la significancia debe considerar el diseño muestral (estrato y UPM de `diseno_muestra`) y no solo `FEX_C`.
3. **Mejorar la reconstrucción del IPM:** incluir el componente de alimentación en primera infancia y revisar el indicador de bajo logro educativo (≈ 40%, más alto de lo esperado).
4. **Evaluar el modelo** con una partición entrenamiento/prueba (AUC y matriz de confusión).

## 9. MODELADO CON DISEÑO MUESTRAL (versión final)

La Sección 8 dejó planteado el siguiente paso: **modelar la probabilidad de que un hogar sea pobre usando solo variables externas al IPM**. Esta sección lo hace y responde, una por una, las 5 recomendaciones del profesor:

| # | Recomendación | Dónde se resuelve |
|---|---|---|
| 1 | Incorporar el diseño muestral (estratos y conglomerados), no solo `FEX_C` | 9.1 y 9.2 |
| 2 | Cuidado con el ingreso per cápita: comparar el modelo con y sin ingreso | 9.3 |
| 3 | Modelo multinivel (hogares dentro de departamentos) o interacción zona × región | 9.5 y 9.6 |
| 4 | Modelo complementario sobre `SCORE_IPM` continuo | 9.7 |
| 5 | Tamaños de efecto y efectos marginales en puntos porcentuales | 9.4 (y en cada modelo) |

**Idea central, en simple.** Antes se usó `FEX_C` para que cada hogar de la muestra "represente" a los hogares que le corresponden en el país. Eso arregla los *porcentajes*, pero no la *incertidumbre*: la encuesta no escogió hogares sueltos, escogió **zonas (segmentos)** y dentro de cada una varios hogares parecidos entre sí. Los hogares de un mismo segmento aportan menos información que hogares independientes, así que los intervalos de confianza de un modelo común salen **demasiado angostos**. Aquí se corrige eso.

In [ ]:
# 9.1 DISEÑO MUESTRAL: estratos y conglomerados
# Qué hace: agrega a la base de hogares las tres variables del diseño de la ECV (estrato de muestreo, municipio y segmento)
#           y arma la UPM (unidad primaria de muestreo = municipio + segmento).
# Por qué: la varianza correcta de un modelo con encuesta depende de estratos y UPM, no solo de FEX_C.
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from types import SimpleNamespace
from patsy import dmatrices, build_design_matrices
from scipy import stats
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

# Rutas (si se corre esta sección sola, se vuelven a definir; en Colab BASE_PATH debe ser "/content")
if "BASE_PATH" not in globals():
    BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_CSV   = os.path.join(BASE_PATH, "data", "raw", "csv")
PROCESSED = os.path.join(BASE_PATH, "data", "processed")
FIGURES   = os.path.join(BASE_PATH, "figures")
TABLAS    = os.path.join(BASE_PATH, "informe", "tablas")
os.makedirs(FIGURES, exist_ok=True); os.makedirs(TABLAS, exist_ok=True)
LLAVE_HOGAR = ["DIRECTORIO", "SECUENCIA_P"]

# Base de hogares limpia (la misma de la Sección 6)
if "hog" not in globals():
    hog = pd.read_parquet(os.path.join(PROCESSED, "ecv2023_hogares_limpia.parquet"))

# Módulo de diseño muestral: viene por persona; se verifica que sea constante dentro de cada hogar.
if "dfs" in globals():
    ds = dfs["diseno_muestra"].copy()
else:
    ruta = os.path.join(RAW_CSV, "diseno_muestra.csv")
    ds = leer_csv(ruta) if "leer_csv" in globals() else pd.read_csv(ruta, sep=None, engine="python", encoding="latin-1")
ds.columns = ds.columns.str.strip().str.upper()
VARS_DIS = ["MPIO", "SEGMENTO", "ESTRATO2020"]
const = ds.groupby(LLAVE_HOGAR)[VARS_DIS].nunique().max()
print("Máximo de valores distintos por hogar (debe ser 1):", const.to_dict())
ds_h = ds[LLAVE_HOGAR + VARS_DIS].drop_duplicates(LLAVE_HOGAR)

# Une al hogar y arma la UPM. Control: no se pierde ni se duplica ningún hogar.
d = hog.merge(ds_h, on=LLAVE_HOGAR, how="left")
assert len(d) == len(hog), "El merge cambió el número de hogares"
print("Hogares sin dato de diseño:", int(d[VARS_DIS].isna().any(axis=1).sum()))
d["PSU"] = d["MPIO"].astype(str) + "_" + d["SEGMENTO"].astype(str)

# Resumen del diseño
n_est = d["ESTRATO2020"].nunique(); n_upm = d["PSU"].nunique()
upm_por_est = d.groupby("ESTRATO2020")["PSU"].nunique()
w = d["FEX_C"]
print(f"\nHogares: {len(d):,} | Estratos de muestreo: {n_est} | UPM (segmentos): {n_upm:,} | Hogares por UPM (promedio): {len(d)/n_upm:.1f}")
print(f"UPM por estrato: mínimo {upm_por_est.min()}, mediana {upm_por_est.median():.0f}, máximo {upm_por_est.max()}")
print(f"Estratos con una sola UPM: {(upm_por_est == 1).sum()}")
print(f"Grados de libertad del diseño (UPM - estratos): {n_upm - n_est:,}")
print(f"Efecto de los pesos (Kish): 1 + CV² = {1 + (w.std()/w.mean())**2:.2f}  -> muestra efectiva por pesos ≈ {w.sum()**2/(w**2).sum():,.0f} hogares")

**Cómo leer 9.1.** Los estratos son grupos de zonas parecidas; las UPM son las zonas (segmentos) que se sortearon dentro de cada estrato. Lo que importa para la varianza es cuántas UPM hay (no cuántos hogares): los hogares de una misma UPM se parecen, así que no cuentan como hogares independientes. Un estrato con una sola UPM no permite estimar variabilidad dentro del estrato; se trata como "certeza" (aporta 0 a la varianza), que es la opción estándar y la que se reporta.

In [ ]:
# 9.2 VARIABLES DEL MODELO Y FUNCIONES DE ESTIMACIÓN CON DISEÑO MUESTRAL
# Qué hace: (a) crea las variables explicativas, todas EXTERNAS al IPM; (b) define svy_glm(), que ajusta un GLM con FEX_C y calcula
#           errores estándar con estratos y UPM (linealización de Taylor); (c) define ame() y wald().
# Por qué: statsmodels no trae errores estándar para encuestas complejas; se implementan aquí y se validan contra otras herramientas.

# (a) Variables. Nada de I01-I15 ni años de educación del jefe: son parte del IPM y dejarían un modelo mecánicamente circular.
REGIONES = {1: "Caribe", 2: "Oriental", 3: "Central", 4: "Pacífica", 5: "Bogotá", 6: "Antioquia",
            7: "Valle", 8: "San Andrés", 9: "Orinoquía-Amazonía"}
REF_REGION = 3     # región de comparación (cámbiala si el profesor prefiere otra)

# Estrato socioeconómico como categoría; los hogares sin estrato (no tienen factura de servicios) se conservan como "Sin estrato".
d["ESTRATO_CAT"] = d["ESTRATO"].map(lambda v: "Sin estrato" if pd.isna(v) else str(int(v)))

# Ingreso per cápita en quintiles ponderados (los hogares con ingreso 0 caen en Q1; sin dato = "Sin dato").
def cuantiles_pond(x, w, qs):
    o = np.argsort(x); x, w = np.asarray(x)[o], np.asarray(w)[o]
    c = np.cumsum(w) / w.sum()
    return [x[np.searchsorted(c, q)] for q in qs]
ok = d["PERCAPITA"].notna()
cortes = cuantiles_pond(d.loc[ok, "PERCAPITA"].values, d.loc[ok, "FEX_C"].values, [.2, .4, .6, .8])
d["Q_ING"] = "Sin dato"
d.loc[ok, "Q_ING"] = ["Q" + str(1 + sum(v > c for c in cortes)) for v in d.loc[ok, "PERCAPITA"]]
print("Cortes de quintiles de ingreso per cápita:", [round(c) for c in cortes])
print(d["Q_ING"].value_counts().sort_index().to_string())
print("Hogares sin estrato:", (d["ESTRATO_CAT"] == "Sin estrato").sum(), "| CLASE:", d["CLASE"].value_counts().to_dict())

# Interacción zona x región: una variable "rural en la región r" solo para regiones que tienen hogares urbanos Y rurales.
tab = pd.crosstab(d["REGION"], d["CLASE"])
REGS_OK = [r for r in tab.index if tab.loc[r].get(1, 0) >= 30 and tab.loc[r, [c for c in tab.columns if c != 1]].sum() >= 30]
REF_INT = REF_REGION if REF_REGION in REGS_OK else REGS_OK[0]
REGS_INT = [r for r in REGS_OK if r != REF_INT]
print("Regiones con hogares urbanos y rurales (para la interacción):", [REGIONES[r] for r in REGS_OK])
def construir(df):
    """Agrega las variables de interacción rural x región (se recalculan cuando se cambia CLASE o REGION)."""
    df = df.copy()
    for r in REGS_INT:
        df[f"RxR{r}"] = ((df["CLASE"] != 1) & (df["REGION"] == r)).astype(int)
    return df
D = construir(d)

BASE = "C(CLASE) + C(REGION, Treatment(reference=%d)) + C(SEXO_JEFE) + EDAD_JEFE + TAM_HOGAR + C(ESTRATO_CAT, Treatment(reference='1'))" % REF_REGION
ING  = " + C(Q_ING, Treatment(reference='Q1'))"

# (b) GLM con varianza de diseño
def svy_glm(formula, data, family=None, w="FEX_C", estrato="ESTRATO2020", upm="PSU", diseno=True):
    """GLM ponderado (pesos normalizados) con covarianza sándwich de Taylor por estrato y UPM.
    diseno=False trata cada hogar como independiente (solo para comparar: error estándar 'ingenuo')."""
    family = family or sm.families.Binomial()
    y, X = dmatrices(formula, data, return_type="dataframe")
    sub = data.loc[y.index]
    wt = sub[w].values.astype(float); wt = wt / wt.mean()
    yy = y.values.ravel(); XX = X.values
    fit = sm.GLM(yy, XX, family=family, freq_weights=wt).fit()
    mu = fit.mu
    A = (XX * (wt * mu * (1 - mu))[:, None]).T @ XX            # "pan" del sándwich
    U = XX * (wt * (yy - mu))[:, None]                           # contribución de cada hogar al score
    if diseno: s, p = sub[estrato].values, sub[upm].values
    else:      s, p = np.zeros(len(sub), dtype=int), np.arange(len(sub))
    Z = pd.DataFrame(U).groupby([s, p]).sum()                    # totales por UPM
    meat = np.zeros((XX.shape[1],) * 2); solitarias = 0
    for _, g in Z.groupby(level=0):
        n = len(g)
        if n < 2: solitarias += 1; continue                      # estrato con una UPM: aporta 0 ("certeza")
        zc = g.values - g.values.mean(0)
        meat += n / (n - 1) * zc.T @ zc
    Ai = np.linalg.inv(A); cov = Ai @ meat @ Ai
    n_upm_, n_est_ = Z.shape[0], Z.index.get_level_values(0).nunique()
    gl = (n_upm_ - n_est_) if diseno else (len(yy) - XX.shape[1])
    nombres = list(X.columns)
    return SimpleNamespace(beta=pd.Series(fit.params, nombres), cov=pd.DataFrame(cov, nombres, nombres),
                           se=pd.Series(np.sqrt(np.diag(cov)), nombres), df=gl, nobs=len(yy), rows=y.index,
                           design_info=X.design_info, mu=mu, y=yy, w=wt, n_upm=n_upm_, n_est=n_est_, solitarias=solitarias)

# (c) Efectos marginales promedio (AME) en puntos porcentuales, con método delta
def ame(res, data, col, nuevo=None, base=None, delta=None, mask=None):
    """Cambio promedio en la probabilidad (pp) si se pasa de 'base' a 'nuevo' en 'col' (o se suma 'delta'),
    manteniendo todo lo demás como está en cada hogar. Error estándar por método delta con la covarianza de diseño."""
    x = data.loc[res.rows]
    if mask is not None: x = x[mask(x)]
    x0, x1 = x.copy(), x.copy()
    if delta is not None: x1[col] = x1[col] + delta
    else:
        if base is not None: x0[col] = base
        x1[col] = nuevo
    X0 = build_design_matrices([res.design_info], construir(x0), return_type="dataframe")[0].values
    X1 = build_design_matrices([res.design_info], construir(x1), return_type="dataframe")[0].values
    b = res.beta.values
    p0, p1 = 1 / (1 + np.exp(-X0 @ b)), 1 / (1 + np.exp(-X1 @ b))
    wt = x["FEX_C"].values; sw = wt.sum()
    g = np.sum(wt * (p1 - p0)) / sw
    grad = ((wt * p1 * (1 - p1))[:, None] * X1 - (wt * p0 * (1 - p0))[:, None] * X0).sum(0) / sw
    se = float(np.sqrt(grad @ res.cov.values @ grad)); t = stats.t.ppf(.975, res.df)
    return dict(pp=100 * g, se=100 * se, lo=100 * (g - t * se), hi=100 * (g + t * se), p=2 * stats.t.sf(abs(g / se), res.df))

def wald(res, nombres):
    """Prueba conjunta de que varios coeficientes son 0 (F de Wald ajustada por diseño)."""
    i = [res.beta.index.get_loc(n) for n in nombres]; b = res.beta.values[i]; V = res.cov.values[np.ix_(i, i)]
    k = len(i); W = float(b @ np.linalg.solve(V, b)); F = W * (res.df - k + 1) / (res.df * k)
    return dict(k=k, F=F, p=stats.f.sf(F, k, res.df - k + 1))

def lista_efectos(data, con_ingreso):
    E = [dict(label="Zona: rural vs. urbano", col="CLASE", base=1, nuevo=2)]
    for r in sorted(data["REGION"].unique()):
        if r != REF_REGION: E.append(dict(label=f"Región {REGIONES[r]} vs. {REGIONES[REF_REGION]}", col="REGION", base=REF_REGION, nuevo=r))
    E += [dict(label="Jefe mujer vs. hombre", col="SEXO_JEFE", base=1, nuevo=2),
          dict(label="Edad del jefe: +10 años", col="EDAD_JEFE", delta=10),
          dict(label="Tamaño del hogar: +1 persona", col="TAM_HOGAR", delta=1)]
    for e in ["2", "3", "4", "5", "6", "Sin estrato"]:
        if e in set(data["ESTRATO_CAT"]): E.append(dict(label=f"Estrato {e} vs. estrato 1", col="ESTRATO_CAT", base="1", nuevo=e))
    if con_ingreso:
        for q in sorted(set(data["Q_ING"]) - {"Q1"}):
            E.append(dict(label=f"Ingreso per cápita {q} vs. Q1", col="Q_ING", base="Q1", nuevo=q))
    return E

def tabla_ame(res, data, con_ingreso, **kw):
    filas = []
    for e in lista_efectos(data, con_ingreso):
        r = ame(res, data, e["col"], nuevo=e.get("nuevo"), base=e.get("base"), delta=e.get("delta"), **kw)
        filas.append({"Efecto": e["label"], **r})
    return pd.DataFrame(filas).set_index("Efecto")

import re
def limpiar(n):
    n = re.sub(r"C\((\w+)(?:, Treatment\(reference=[^)]*\))?\)\[T\.(.+)\]", r"\1 = \2", n)
    return n.replace("Intercept", "Constante")
print("\nFunciones listas.")

### 9.2b Validación de la varianza de diseño

Como `svy_glm()` es una implementación propia, se comprueba antes de usarla: (i) con **un solo estrato** debe coincidir (salvo el factor de corrección de pocos clusters) con los errores estándar por conglomerado de `statsmodels`; (ii) la librería `samplics` (especializada en encuestas) todavía no implementa la regresión logística, así que la comprobación se hace contra `statsmodels`; el paso extra de `svy_glm` es sumar la varianza por estrato con la fórmula estándar de linealización de Taylor.

In [ ]:
# 9.2b VALIDACIÓN de svy_glm
# Qué hace: compara los errores estándar de svy_glm (con un solo estrato) contra statsmodels con errores por conglomerado.
# Resultado esperado: razones cercanas a 1 (la pequeña diferencia es el factor n/(n-1) de pocos conglomerados).
f_test = "POBRE_IPM ~ C(CLASE) + C(SEXO_JEFE) + EDAD_JEFE + TAM_HOGAR"
T = D.copy(); T["UNO"] = 1
mio = svy_glm(f_test, T, estrato="UNO")
y_, X_ = dmatrices(f_test, T, return_type="dataframe")
wn = T.loc[y_.index, "FEX_C"].values; wn = wn / wn.mean()
sm_cl = sm.GLM(y_.values.ravel(), X_.values, family=sm.families.Binomial(), freq_weights=wn).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(T.loc[y_.index, "PSU"])[0], "use_correction": False})
G = mio.n_upm
chk = pd.DataFrame({"SE svy_glm": mio.se.values, "SE statsmodels (cluster)": sm_cl.bse,
                    "razón (esperada ≈ √(G/(G-1)))": mio.se.values / sm_cl.bse}, index=mio.beta.index)
print(f"UPM (G) = {G:,}  ->  √(G/(G-1)) = {np.sqrt(G/(G-1)):.5f}")
display(chk)
print("\nNota: la librería samplics (SurveyGLM) aún no implementa regresión logística (devuelve 'not implemented yet'), por eso la validación se hace contra statsmodels.")
print("El único paso adicional de svy_glm es sumar la varianza estrato por estrato (fórmula estándar de linealización de Taylor).")

### 9.3 Modelo M1 (sin ingreso) vs. M2 (con ingreso) — Recomendaciones 1 y 2

**Por qué el cuidado con el ingreso.** El ingreso per cápita *parece* un predictor ideal de pobreza, pero (a) el IPM se diseñó justamente para medir lo que el ingreso no captura, así que su relación con el ingreso es de "primo cercano", no de causa y efecto; (b) el ingreso de hogares pobres está muy mal medido (ceros, ingresos informales, hogares que no reportan); y (c) al estar tan correlacionado con estrato, zona y educación, **se lleva el crédito de las demás variables** y puede cambiar su tamaño y hasta su signo. Por eso se ajustan **dos modelos** y se compara qué cambia.

In [ ]:
# 9.3 AJUSTE DE M1 (SIN INGRESO) Y M2 (CON INGRESO)
# Qué hace: ajusta ambos logit con FEX_C y varianza de diseño; compara errores estándar de diseño vs. ingenuos (DEFT) y el AUC ponderado.
# Variable respuesta: POBRE_IPM (1 = hogar pobre según el IPM reconstruido).
t0 = time.time()
M1 = svy_glm("POBRE_IPM ~ " + BASE, D)
M2 = svy_glm("POBRE_IPM ~ " + BASE + ING, D)
M1n = svy_glm("POBRE_IPM ~ " + BASE, D, diseno=False)
M2n = svy_glm("POBRE_IPM ~ " + BASE + ING, D, diseno=False)
print(f"Ajustados en {time.time()-t0:.0f} s | hogares usados M1: {M1.nobs:,}, M2: {M2.nobs:,} | UPM: {M2.n_upm:,} | estratos: {M2.n_est} | estratos con 1 UPM: {M2.solitarias}")

def tabla_coef(res, ingenuo):
    t = stats.t.ppf(.975, res.df)
    out = pd.DataFrame({"OR": np.exp(res.beta), "OR_IC95_inf": np.exp(res.beta - t * res.se), "OR_IC95_sup": np.exp(res.beta + t * res.se),
                        "p": 2 * stats.t.sf(np.abs(res.beta / res.se), res.df),
                        "SE_diseño": res.se, "SE_ingenuo": ingenuo.se, "DEFT": res.se / ingenuo.se})
    out.index = [limpiar(n) for n in out.index]
    return out
OR1, OR2 = tabla_coef(M1, M1n), tabla_coef(M2, M2n)
print("\n--- M2 (con ingreso): odds ratios y efecto del diseño muestral ---")
display(OR2.round(3))
print(f"DEFT (razón de errores estándar diseño/ingenuo): mediana {OR2['DEFT'].median():.2f}, máximo {OR2['DEFT'].max():.2f}  "
      f"-> los intervalos ingenuos eran ~{(OR2['DEFT'].median()-1)*100:.0f}% más angostos de lo que corresponde.")

# Desempeño: AUC ponderado (dentro de muestra; sirve para comparar M1 vs M2, no como medida de predicción futura)
AUC1 = roc_auc_score(M1.y, M1.mu, sample_weight=M1.w); AUC2 = roc_auc_score(M2.y, M2.mu, sample_weight=M2.w)
print(f"\nAUC ponderado  M1 (sin ingreso): {AUC1:.3f} | M2 (con ingreso): {AUC2:.3f} | ganancia por agregar ingreso: {AUC2-AUC1:+.3f}")
wi = [n for n in M2.beta.index if "Q_ING" in n]
W_ing = wald(M2, wi); print(f"Prueba conjunta de ingreso (diseño): F({W_ing['k']}) = {W_ing['F']:.1f}, p = {W_ing['p']:.3g}")
OR1.to_csv(os.path.join(TABLAS, "09_M1_odds_ratios.csv")); OR2.to_csv(os.path.join(TABLAS, "09_M2_odds_ratios.csv"))

### 9.4 Efectos marginales en puntos porcentuales — Recomendación 5

Un odds ratio es difícil de interpretar. El **efecto marginal promedio (AME)** responde algo más directo: *"si todos los hogares fueran rurales en vez de urbanos (y todo lo demás quedara igual), ¿en cuántos puntos porcentuales cambiaría la probabilidad promedio de ser pobre?"*. Se calcula para cada hogar con su propia situación y se promedia con `FEX_C`; el intervalo de confianza usa la varianza de diseño.

In [ ]:
# 9.4 EFECTOS MARGINALES PROMEDIO (pp de probabilidad) EN M1 Y M2, y qué cambia al incluir el ingreso
# Qué hace: calcula el AME de cada variable en M1 y M2, los pone lado a lado y los grafica.
t0 = time.time()
A1 = tabla_ame(M1, D, con_ingreso=False)
A2 = tabla_ame(M2, D, con_ingreso=True)
print(f"AME calculados en {time.time()-t0:.0f} s")
comp = A1[["pp", "lo", "hi"]].join(A2[["pp", "lo", "hi"]], how="outer", lsuffix="_M1", rsuffix="_M2")
comp["cambio_pp"] = comp["pp_M2"] - comp["pp_M1"]
comp["cambio_%"] = 100 * comp["cambio_pp"] / comp["pp_M1"].abs()
print("Efecto marginal promedio en puntos porcentuales de probabilidad de ser pobre (IC 95% con diseño muestral):")
display(comp.round(2))
A1.to_csv(os.path.join(TABLAS, "09_M1_AME_pp.csv")); A2.to_csv(os.path.join(TABLAS, "09_M2_AME_pp.csv")); comp.to_csv(os.path.join(TABLAS, "09_comparacion_M1_M2.csv"))

# Gráfico de bosque: M1 (sin ingreso) vs M2 (con ingreso)
sin_ing = [i for i in comp.index if "Ingreso" not in i]
fig, ax = plt.subplots(figsize=(9, 0.38 * len(comp) + 1.5))
y = np.arange(len(comp))[::-1]
for k, (suf, c, m, off) in enumerate([("M1", "#4C72B0", "o", .13), ("M2", "#DD8452", "s", -.13)]):
    sub = comp[[f"pp_{suf}", f"lo_{suf}", f"hi_{suf}"]]; mk = sub.notna().all(axis=1).values
    ax.errorbar(sub[f"pp_{suf}"][mk], y[mk] + off, xerr=[(sub[f"pp_{suf}"] - sub[f"lo_{suf}"])[mk], (sub[f"hi_{suf}"] - sub[f"pp_{suf}"])[mk]],
                fmt=m, color=c, capsize=2, label="M1: sin ingreso" if suf == "M1" else "M2: con ingreso")
ax.axvline(0, color="gray", lw=1); ax.set_yticks(y); ax.set_yticklabels(comp.index, fontsize=8)
ax.set_xlabel("Cambio en la probabilidad de ser pobre (puntos porcentuales)"); ax.set_title("Efectos marginales promedio (IC 95%, diseño muestral)")
ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES, "09_efectos_marginales_M1_M2.png"), dpi=150); plt.show()

**Qué mirar en la comparación M1 vs. M2 (completar con los números al ejecutar).** (1) Si el efecto de *zona rural* y de *estrato* se **reduce mucho** al agregar ingreso, parte de lo que parecía "efecto del territorio" era "efecto de tener o no ingreso". (2) Si el AUC sube poco, el ingreso aporta poca información nueva sobre la pobreza multidimensional. (3) El mensaje para la sustentación: el ingreso **no reemplaza** al IPM; por eso el modelo principal para interpretar es el M1 y el M2 se usa como análisis de sensibilidad.

### 9.5 Interacción zona × región — Recomendación 3 (opción A)

La brecha rural–urbano probablemente **no es igual en todo el país**. Se agrega al M2 una interacción: un término de "rural" por cada región. Si la prueba conjunta es significativa, el efecto de la zona **depende** de la región, y entonces conviene reportarlo región por región.

In [ ]:
# 9.5 INTERACCIÓN ZONA x REGIÓN
# Qué hace: agrega al M2 los términos "rural x región" (solo regiones con hogares urbanos y rurales), prueba si son necesarios
#           y calcula la brecha rural-urbano (en pp) dentro de cada región.
f3 = "POBRE_IPM ~ " + BASE + ING + " + " + " + ".join(f"RxR{r}" for r in REGS_INT)
M3 = svy_glm(f3, D)
Wx = wald(M3, [f"RxR{r}" for r in REGS_INT])
print(f"Prueba conjunta de la interacción zona x región (diseño): F({Wx['k']}) = {Wx['F']:.2f}, p = {Wx['p']:.3g}")
print(f"AUC ponderado M3: {roc_auc_score(M3.y, M3.mu, sample_weight=M3.w):.3f} (M2: {AUC2:.3f})")

filas = []
tot = ame(M3, D, "CLASE", nuevo=2, base=1)
filas.append({"Región": "Nacional (promedio)", **tot})
for r in REGS_OK:
    filas.append({"Región": REGIONES[r], **ame(M3, D, "CLASE", nuevo=2, base=1, mask=lambda x, r=r: x["REGION"] == r)})
brecha = pd.DataFrame(filas).set_index("Región")
print("\nBrecha rural vs. urbano por región (pp de probabilidad de ser pobre, IC 95%):")
display(brecha.round(2))
sin = [REGIONES[r] for r in sorted(set(D['REGION'])) if r not in REGS_OK]
if sin: print("Regiones sin hogares de ambas zonas (no se estima brecha):", sin)
brecha.to_csv(os.path.join(TABLAS, "09_M3_brecha_rural_urbano_por_region.csv"))

fig, ax = plt.subplots(figsize=(7, 0.45 * len(brecha) + 1.2)); yy = np.arange(len(brecha))[::-1]
ax.errorbar(brecha["pp"], yy, xerr=[brecha["pp"] - brecha["lo"], brecha["hi"] - brecha["pp"]], fmt="o", color="#4C72B0", capsize=3)
ax.axvline(0, color="gray", lw=1); ax.set_yticks(yy); ax.set_yticklabels(brecha.index)
ax.set_xlabel("Brecha rural – urbano (pp de probabilidad de ser pobre)"); ax.set_title("La brecha rural-urbano por región (M3, IC 95%)")
plt.tight_layout(); plt.savefig(os.path.join(FIGURES, "09_brecha_rural_urbano_region.png"), dpi=150); plt.show()

### 9.6 Modelo multinivel: hogares dentro de departamentos — Recomendación 3 (opción B)

Un modelo multinivel le da a **cada departamento su propio nivel base de pobreza** (un "intercepto aleatorio") en vez de forzar que las regiones, que son grupos grandes, expliquen todo. Resultado clave: **σ**, cuánto varían los departamentos entre sí, y el **ICC**, el % de la variación total en pobreza que queda *entre* departamentos.

*Limitación honesta:* `statsmodels` no admite pesos de muestreo en modelos mixtos, así que este modelo es **sin ponderar** y se usa para entender la estructura (cuánto importa el departamento), no para estimar niveles nacionales. Los efectos "fijos" se contrastan con el logit sin ponderar para ver que no cambian las conclusiones.

In [ ]:
# 9.6 MODELO MULTINIVEL (intercepto aleatorio por departamento)
# Qué hace: ajusta (a) M2 sin región + intercepto por departamento y (b) M2 con región + intercepto por departamento.
# Cómo leer: sigma = desviación estándar entre departamentos (escala logit); ICC = sigma^2 / (sigma^2 + pi^2/3).
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
cols_ml = ["POBRE_IPM", "CLASE", "REGION", "SEXO_JEFE", "EDAD_JEFE", "TAM_HOGAR", "ESTRATO_CAT", "Q_ING", "P1_DEPARTAMENTO"]
Dm = D.dropna(subset=cols_ml).copy()
Dm["POBRE_IPM"] = Dm["POBRE_IPM"].astype(int)
f_ml_a = "POBRE_IPM ~ C(CLASE) + C(SEXO_JEFE) + EDAD_JEFE + TAM_HOGAR + C(ESTRATO_CAT, Treatment(reference='1')) + C(Q_ING, Treatment(reference='Q1'))"
f_ml_b = f_ml_a + " + C(REGION, Treatment(reference=%d))" % REF_REGION
ML = {}
for nombre, f in [("Sin región", f_ml_a), ("Con región", f_ml_b)]:
    t0 = time.time()
    m = BinomialBayesMixedGLM.from_formula(f, {"dep": "0 + C(P1_DEPARTAMENTO)"}, data=Dm)
    r = m.fit_vb()
    s = float(np.exp(r.vcp_mean[0])); lo, hi = np.exp(r.vcp_mean[0] - 1.96 * r.vcp_sd[0]), np.exp(r.vcp_mean[0] + 1.96 * r.vcp_sd[0])
    ML[nombre] = dict(res=r, sigma=s, lo=lo, hi=hi, icc=s**2 / (s**2 + np.pi**2 / 3), seg=time.time() - t0)
    print(f"Multinivel ({nombre}): sigma = {s:.2f} (IC95% aprox. {lo:.2f}-{hi:.2f}) | ICC latente = {ML[nombre]['icc']*100:.1f}% | {len(Dm):,} hogares, {Dm['P1_DEPARTAMENTO'].nunique()} departamentos | {ML[nombre]['seg']:.0f} s")

# Efectos fijos: multinivel vs. logit sin ponderar (log-odds). Deben ser parecidos.
rb = ML["Con región"]["res"]
fe = pd.DataFrame({"multinivel": rb.fe_mean, "ee_multinivel": rb.fe_sd}, index=rb.model.exog_names)
glm_np = sm.GLM(Dm["POBRE_IPM"], build_design_matrices([dmatrices(f_ml_b, Dm, return_type="dataframe")[1].design_info], Dm, return_type="dataframe")[0],
                family=sm.families.Binomial()).fit()
fe["logit_sin_pesos"] = glm_np.params.values
fe.index = [limpiar(n) for n in fe.index]
display(fe.round(3))
fe.to_csv(os.path.join(TABLAS, "09_multinivel_efectos_fijos.csv"))
print("Lectura: si 'sigma' baja mucho al agregar REGION, gran parte de la diferencia entre departamentos ya la explican las regiones.")

### 9.7 Modelo complementario sobre `SCORE_IPM` continuo — Recomendación 4

`POBRE_IPM` convierte el puntaje en "pobre / no pobre" y **tira información**: un hogar con puntaje 0,32 (justo debajo del umbral) cuenta igual que uno con 0. El modelo complementario usa el puntaje completo (0–1, con muchos ceros). Se usa **regresión logit fraccional** (cuasi-verosimilitud binomial con respuesta entre 0 y 1) en lugar de beta, porque la beta exige valores estrictamente entre 0 y 1 y aquí hay ceros exactos. El efecto marginal se lee en **puntos del puntaje** (0–100): lo que sube o baja la privación promedio del hogar.

In [ ]:
# 9.7 REGRESIÓN LOGIT FRACCIONAL SOBRE SCORE_IPM
# Qué hace: ajusta el mismo M2 pero con SCORE_IPM como respuesta, con varianza de diseño, y compara con el efecto sobre POBRE_IPM.
print("SCORE_IPM: mínimo", D["SCORE_IPM"].min(), "| máximo", D["SCORE_IPM"].max(), "| % de hogares con puntaje 0:", round((D["SCORE_IPM"] == 0).mean() * 100, 1))
assert D["SCORE_IPM"].between(0, 1).all(), "SCORE_IPM debe estar entre 0 y 1"
MF = svy_glm("SCORE_IPM ~ " + BASE + ING, D)
AF = tabla_ame(MF, D, con_ingreso=True)   # pp de la escala 0-1 = puntos de puntaje (0-100)
wm = np.average(MF.mu, weights=MF.w); wy = np.average(MF.y, weights=MF.w)
r2 = (np.cov(MF.y, MF.mu, aweights=MF.w)[0, 1]) ** 2 / (np.cov(MF.y, MF.y, aweights=MF.w)[0, 0] * np.cov(MF.mu, MF.mu, aweights=MF.w)[0, 0])
print(f"R² ponderado (correlación² entre puntaje observado y predicho): {r2:.3f}")
cmp7 = A2[["pp"]].rename(columns={"pp": "pp_prob_pobre (logit)"}).join(AF[["pp", "lo", "hi"]].rename(columns={"pp": "puntos_score (fraccional)", "lo": "lo_score", "hi": "hi_score"}))
display(cmp7.round(2))
cmp7.to_csv(os.path.join(TABLAS, "09_fraccional_vs_logit_AME.csv"))
print("Lectura: si los signos y el orden de importancia coinciden con el logit, las conclusiones no dependen de haber usado el umbral del 33,3%.")

### 9.8 Resumen de resultados para la sustentación y para sincronizar ficha y trabajo escrito

In [ ]:
# 9.8 RESUMEN DE RESULTADOS (copiar y pegar en el chat para actualizar la ficha, el Word y el README)
print("="*80); print("RESUMEN PARA PEGAR"); print("="*80)
print(f"Diseño: {M2.nobs:,} hogares | {M2.n_est} estratos | {M2.n_upm:,} UPM | gl = {M2.df:,} | estratos con 1 UPM: {M2.solitarias}")
print(f"DEFT mediana {OR2['DEFT'].median():.2f}, máximo {OR2['DEFT'].max():.2f}")
print(f"AUC ponderado: M1 {AUC1:.3f} | M2 {AUC2:.3f} | M3 {roc_auc_score(M3.y, M3.mu, sample_weight=M3.w):.3f}")
print(f"Interacción zona x región: F({Wx['k']}) = {Wx['F']:.2f}, p = {Wx['p']:.3g}")
print(f"Multinivel: sigma sin región {ML['Sin región']['sigma']:.2f} (ICC {ML['Sin región']['icc']*100:.1f}%) | con región {ML['Con región']['sigma']:.2f} (ICC {ML['Con región']['icc']*100:.1f}%)")
print(f"Fraccional: R² ponderado {r2:.3f}")
print("\nEfectos marginales (pp), M1 vs M2:"); print(comp[["pp_M1", "lo_M1", "hi_M1", "pp_M2", "lo_M2", "hi_M2"]].round(1).to_string())
print("\nBrecha rural-urbano por región (pp):"); print(brecha[["pp", "lo", "hi"]].round(1).to_string())
print("\nFraccional (puntos de puntaje):"); print(AF[["pp", "lo", "hi"]].round(2).to_string())
print("\nTablas guardadas en:", TABLAS, "| Figuras en:", FIGURES)